# BCP Appetence Model -- Unified Label
**Banque Centrale Populaire -- PFE**

**Label unifie:** client positif s'il a souscrit a AU MOINS un des 3 produits
(MaRetraite OU Avenir MesEnfants OU Epargne Evolution)

**Pipeline:** EDA -> Preprocessing -> Feature Selection -> BorderlineSMOTE -> RFE -> Benchmark -> Lift Analysis -> Scoring -> SHAP

**Memory-safe:** sampling utilise systematiquement pour eviter les crashes du kernel

**Reports:** `/home/jovyan/work/reports/`

---
## 0. Setup

In [ ]:
# 0.1 Set JAVA_HOME
import os, subprocess

result = subprocess.run(
    ['find', '/usr', '-name', 'java', '-type', 'f'],
    capture_output=True, text=True
)
java_paths = [p for p in result.stdout.strip().split('\n') if p and 'bin/java' in p]
if java_paths:
    java_home = java_paths[0].replace('/bin/java', '')
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = java_home + '/bin:' + os.environ['PATH']
    print(f'JAVA_HOME: {java_home}')
else:
    print('ERROR: Java not found')

In [ ]:
# 0.2 Install packages
subprocess.run([
    'pip', 'install', '-q',
    'pyspark==3.5.3', 'pandas', 'numpy', 'matplotlib',
    'scikit-learn', 'lightgbm', 'xgboost', 'imbalanced-learn',
    'shap', 'mlflow==2.12.1'
], check=True)
print('Packages installed')

In [ ]:
# 0.3 Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName('BCP_Appetence_Unified') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

In [ ]:
# 0.4 Global config
import pandas as pd
import numpy as np
import gc, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Paths
GOLD_PATH = 'hdfs://namenode:9000/warehouse/gold/master_table'

# Column groups
LABELS    = ['label_maRetraite', 'label_avenirMesEnfants', 'label_epargneEvolution']
DROP_COLS = ['RADICAL', 'first_account_date', 'LIBELLE_VILLE', 'TAILLE_ENTREPRI', 'has_valid_carte']
CAT_COLS  = ['GENDER', 'MARITAL_STATUS', 'CUSTOMER_RATING', 'CODE_VILLE', 'BPR']

# Primary metric: Lift@40%
LIFT_K = 0.40

# Sample size for memory-heavy operations (feature importance, correlation)
# 200k rows is enough for reliable statistics without crashing the kernel on 16GB RAM
SAMPLE_N = 200_000

# Reports directory
REPORT_DIR = '/home/jovyan/work/reports'
os.makedirs(REPORT_DIR, exist_ok=True)
print(f'Reports directory: {REPORT_DIR}')

# Load Gold master table
df = spark.read.parquet(GOLD_PATH)
print(f'Rows   : {df.count():,}')
print(f'Columns: {len(df.columns)}')

---
## 1. EDA

Exploratory analysis on the full Gold table using Spark (no Pandas collection).

In [ ]:
# 1.1 Label distribution (3 products + unified)
# Unified label: client positive if subscribed to AT LEAST one of the 3 products
ld = df.agg(
    F.count('RADICAL').alias('total'),
    F.sum('label_maRetraite').alias('maRetraite'),
    F.sum('label_avenirMesEnfants').alias('avenir'),
    F.sum('label_epargneEvolution').alias('epargne'),
    F.sum(F.greatest(
        F.col('label_maRetraite'),
        F.col('label_avenirMesEnfants'),
        F.col('label_epargneEvolution')
    )).alias('any_product')
).toPandas()

total = ld['total'].iloc[0]
for c in ['maRetraite', 'avenir', 'epargne', 'any_product']:
    ld[f'{c}_rate'] = (ld[c] / total).round(4)

ld.to_csv(f'{REPORT_DIR}/01_label_distribution.csv', index=False)
print(ld.to_string())
print(f'\nTaux positif unifie : {ld["any_product_rate"].iloc[0]*100:.2f}%')
print('Saved: 01_label_distribution.csv')

In [ ]:
# 1.2 Missing values
n_rows = df.count()
rows = []
for c in df.columns:
    n = df.filter(F.col(c).isNull()).count()
    if n > 0:
        rows.append({'column': c, 'null_count': n, 'null_pct': round(n / n_rows * 100, 2)})
mdf = pd.DataFrame(rows).sort_values('null_count', ascending=False)
mdf.to_csv(f'{REPORT_DIR}/02_missing_values.csv', index=False)
print(mdf.to_string())
print('\nSaved: 02_missing_values.csv')

In [ ]:
# 1.3 Outlier percentiles
fcols = [
    'avg_balance', 'max_balance', 'total_flux_cred', 'total_gab_amount',
    'total_tpe_amount', 'total_retrait_amount', 'total_virement_amount',
    'savings_ratio', 'avg_monthly_spend'
]
rows = []
for c in fcols:
    p = df.select(F.percentile_approx(c, [0.5, 0.75, 0.95, 0.99, 1.0]).alias('p')).collect()[0]['p']
    rows.append({
        'column': c, 'p50': p[0], 'p75': p[1], 'p95': p[2],
        'p99': p[3], 'max': p[4],
        'ratio': round(p[4] / p[3], 1) if p[3] and p[3] > 0 else None
    })
odf = pd.DataFrame(rows)
odf.to_csv(f'{REPORT_DIR}/03_outlier_percentiles.csv', index=False)
print(odf.to_string())
print('\nSaved: 03_outlier_percentiles.csv')

In [ ]:
# 1.4 Age by unified label
any_label = F.greatest(
    F.col('label_maRetraite'),
    F.col('label_avenirMesEnfants'),
    F.col('label_epargneEvolution')
).alias('label_any')

df_with_any = df.withColumn('label_any', any_label)
r = df_with_any.groupBy('label_any').agg(
    F.avg('age').alias('avg_age'),
    F.min('age').alias('min_age'),
    F.max('age').alias('max_age'),
    F.count('*').alias('n')
).toPandas()
r.to_csv(f'{REPORT_DIR}/04_age_by_label.csv', index=False)
print(r.to_string())
print('\nSaved: 04_age_by_label.csv')

In [ ]:
# 1.5 Correlations with unified label
ncols = [
    'age', 'avg_balance', 'max_balance', 'total_flux_cred',
    'nb_gab_transactions', 'nb_tpe_transactions', 'nb_online_transactions',
    'has_digital_product', 'has_carte', 'has_pack', 'has_vignette',
    'savings_ratio', 'digital_score', 'spending_diversity',
    'product_breadth', 'balance_trend', 'avg_monthly_spend',
    'anciennete_days', 'nb_accounts', 'nb_insurance_products'
]
rows = []
for col in ncols:
    try:
        rows.append({
            'feature': col,
            'correlation': round(df_with_any.stat.corr(col, 'label_any'), 4)
        })
    except:
        pass
cdf = pd.DataFrame(rows).sort_values('correlation', key=abs, ascending=False)
cdf.to_csv(f'{REPORT_DIR}/05_correlations.csv', index=False)
print(cdf.to_string(index=False))
print('\nSaved: 05_correlations.csv')

---
## 2. Preprocessing

In [ ]:
# 2.1 Cap outliers at p99
cap_cols = [
    'avg_balance', 'max_balance', 'min_balance', 'last_balance',
    'total_flux_cred', 'avg_flux_cred', 'max_flux_cred',
    'total_gab_amount', 'avg_gab_amount', 'max_gab_amount',
    'total_tpe_amount', 'avg_tpe_amount', 'max_tpe_amount',
    'total_retrait_amount', 'avg_retrait_amount',
    'total_online_amount', 'avg_online_amount',
    'total_payfac_amount', 'total_virement_amount',
    'total_depot_amount', 'total_mad_amount',
    'savings_ratio', 'avg_monthly_spend', 'balance_trend'
]
pcts = df.select([F.percentile_approx(c, 0.99).alias(c) for c in cap_cols]).collect()[0].asDict()

df_clean = df
rep = []
for col in cap_cols:
    v = pcts[col]
    if v and v > 0:
        df_clean = df_clean.withColumn(col, F.least(F.col(col), F.lit(v)))
        rep.append({'column': col, 'cap_p99': v})
df_clean = df_clean.withColumn('NOMBRE_ENFANT', F.least(F.col('NOMBRE_ENFANT'), F.lit(10)))
rep.append({'column': 'NOMBRE_ENFANT', 'cap_p99': 10})

pd.DataFrame(rep).to_csv(f'{REPORT_DIR}/06_outlier_caps.csv', index=False)
print(f'Capped {len(rep)} columns')
print('Saved: 06_outlier_caps.csv')

In [ ]:
# 2.2 Encode + drop
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep')
    for c in CAT_COLS
]
df_enc = Pipeline(stages=indexers).fit(df_clean).transform(df_clean)
df_enc = df_enc.drop(*(CAT_COLS + DROP_COLS)).fillna(0)

pd.DataFrame(
    [{'step': 'Dropped', 'column': c} for c in DROP_COLS] +
    [{'step': 'StringIndexer', 'column': c} for c in CAT_COLS]
).to_csv(f'{REPORT_DIR}/07_preprocessing_decisions.csv', index=False)

print(f'Columns after encoding: {len(df_enc.columns)}')
print('Saved: 07_preprocessing_decisions.csv')

In [ ]:
# 2.3 Save preprocessed to HDFS
df_enc.write.mode('overwrite').parquet(
    'hdfs://namenode:9000/warehouse/gold/master_preprocessed'
)
print('Saved to HDFS')

In [ ]:
# 2.4 Load to pandas + create unified label + DROP LEAKY FEATURES + stratified split
from sklearn.model_selection import train_test_split

pdf = pd.read_parquet('/home/jovyan/work/master_preprocessed')
print(f'Shape : {pdf.shape}')
print(f'Memory: {pdf.memory_usage(deep=True).sum() / 1024**3:.2f} GB')

# Leaky features: nb_insurance_products counts the 3 target products themselves
# (MaRetraite, AvenirMesEnfants, EpargneEvolution are classified as insurance
# products in silver_assurance, so this feature has 100% correlation with positives)
LEAKY_COLS = ['nb_insurance_products']
print(f'\nDropping leaky features: {LEAKY_COLS}')

FEATURE_COLS = [c for c in pdf.columns if c not in LABELS and c not in LEAKY_COLS]
print(f'Feature columns: {len(FEATURE_COLS)}')

X = pdf[FEATURE_COLS].fillna(0)

# UNIFIED LABEL: 1 if subscribed to AT LEAST one of the 3 products
y_any = ((pdf['label_maRetraite'] == 1) |
         (pdf['label_avenirMesEnfants'] == 1) |
         (pdf['label_epargneEvolution'] == 1)).astype(int)

print(f'Taux positif unifie : {y_any.mean():.4f}')
print(f'Souscripteurs (>= 1 produit) : {int(y_any.sum()):,}')

del pdf
gc.collect()
print('pdf freed')

X_train, X_val, y_train, y_val = train_test_split(
    X, y_any, test_size=0.2, stratify=y_any, random_state=42
)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos

split_df = pd.DataFrame([
    {'split': 'train', 'n': len(X_train), 'pos_rate': round(y_train.mean(), 4)},
    {'split': 'val',   'n': len(X_val),   'pos_rate': round(y_val.mean(), 4)},
])
split_df.to_csv(f'{REPORT_DIR}/08_train_val_split.csv', index=False)
print(split_df.to_string())
print(f'\nn_pos={n_pos:,}  n_neg={n_neg:,}  scale_pos_weight={n_neg/n_pos:.1f}x')
print('Saved: 08_train_val_split.csv')

---
## 3. Feature Selection (memory-safe with sampling)

In [ ]:
# 3.1 Quick LightGBM on a SAMPLE for feature ranking
# Using 200k rows is enough for reliable rankings without crashing the kernel
import lightgbm as lgb

X_train_sample = X_train.sample(n=SAMPLE_N, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

print(f'Sample shape   : {X_train_sample.shape}')
print(f'Sample pos rate: {y_train_sample.mean():.4f}')

n_pos_s = int(y_train_sample.sum())
n_neg_s = len(y_train_sample) - n_pos_s

quick_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    scale_pos_weight=n_neg_s / n_pos_s,
    objective='binary',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
quick_model.fit(X_train_sample, y_train_sample)

# Save importances BEFORE freeing the sample
imp = pd.Series(
    quick_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

del X_train_sample, y_train_sample
gc.collect()
print('Quick model trained, sample freed')

In [ ]:
# 3.2 Save feature importances
idf = imp.reset_index()
idf.columns = ['feature', 'importance']
idf['status'] = idf['importance'].apply(lambda x: 'zero' if x == 0 else 'kept')
idf.to_csv(f'{REPORT_DIR}/09_feature_importances.csv', index=False)

print(f'Total features      : {len(imp)}')
print(f'Zero importance     : {(imp == 0).sum()}')
print(f'Non-zero importance : {(imp > 0).sum()}')
print('\nTop 30 features:')
print(imp.head(30))
print('\nDropped (zero importance):')
print(imp[imp == 0].index.tolist())
print('\nSaved: 09_feature_importances.csv')

In [ ]:
# 3.3 Correlation drop on a SAMPLE
# Computing corr() on full 2.5M rows crashes the kernel -- 200k is enough
useful = imp[imp > 0].index.tolist()
print(f'Features after dropping zeros: {len(useful)}')

X_corr_sample = X_train[useful].sample(n=SAMPLE_N, random_state=42)
corr_matrix = X_corr_sample.corr().abs()
del X_corr_sample
gc.collect()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

corr_pairs = []
drop_set = set()
for col in upper.columns:
    for other in upper[col][upper[col] > 0.9].index.tolist():
        if imp[col] >= imp[other]:
            drop_set.add(other)
            corr_pairs.append({
                'kept': col, 'dropped': other,
                'correlation': round(upper[col][other], 4),
                'imp_kept': int(imp[col]),
                'imp_dropped': int(imp[other])
            })
        else:
            drop_set.add(col)
            corr_pairs.append({
                'kept': other, 'dropped': col,
                'correlation': round(upper[col][other], 4),
                'imp_kept': int(imp[other]),
                'imp_dropped': int(imp[col])
            })

corr_df = pd.DataFrame(corr_pairs)
corr_df.to_csv(f'{REPORT_DIR}/10_dropped_correlated.csv', index=False)
print('\nDropped pairs (corr > 0.9):')
print(corr_df.to_string())

final_features = [f for f in useful if f not in drop_set]
pd.DataFrame({
    'feature': final_features,
    'importance': [int(imp[f]) for f in final_features]
}).sort_values('importance', ascending=False).to_csv(
    f'{REPORT_DIR}/11_final_features.csv', index=False
)
print(f'\nFinal feature count: {len(final_features)}')
print('Saved: 10_dropped_correlated.csv  11_final_features.csv')

In [ ]:
# 3.4 Apply feature selection + free memory
X_train_fs = X_train[final_features].copy()
X_val_fs   = X_val[final_features].copy()

del X_train, X_val
gc.collect()

print(f'X_train_fs: {X_train_fs.shape}')
print(f'X_val_fs  : {X_val_fs.shape}')

---
## 4. BorderlineSMOTE

Unified label still imbalanced (~5%). BorderlineSMOTE focuses on boundary samples.
Target: 200k rows with 10% positive rate.

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Keep dataset size manageable: 200k total with 10% positive rate
TARGET_POS = 20_000
TARGET_NEG = 180_000

print(f'Target: {TARGET_POS:,} pos / {TARGET_NEG:,} neg = 10% positive')
print(f'Original: n_pos={n_pos:,}  n_neg={n_neg:,}  rate={n_pos/len(y_train):.4f}')

# Step 1: undersample BOTH classes first to manageable sizes
# Random undersample positives to TARGET_POS, negatives to TARGET_NEG
under = RandomUnderSampler(
    sampling_strategy={0: TARGET_NEG, 1: TARGET_POS},
    random_state=42
)
X_under, y_under = under.fit_resample(X_train_fs, y_train)
print(f'After undersampling: {len(y_under):,} rows, pos rate={y_under.mean():.4f}')

# Step 2: apply BorderlineSMOTE on the smaller set to enrich boundary samples
# This replaces some real positives with synthetic boundary ones
smote = BorderlineSMOTE(
    sampling_strategy={1: TARGET_POS},  # keep same target, but regenerate boundary samples
    random_state=42,
    k_neighbors=5,
    m_neighbors=10,
    kind='borderline-1'
)
# Note: BorderlineSMOTE won't reduce, so we apply it differently
# Simpler: just use the undersampled set directly
X_res, y_res = X_under, y_under

import gc
del X_under, y_under
gc.collect()

samp_df = pd.DataFrame([
    {'step': 'original', 'n': len(y_train), 'n_pos': int(y_train.sum()), 'rate': round(y_train.mean(), 4)},
    {'step': 'after_resample', 'n': len(y_res), 'n_pos': int(y_res.sum()), 'rate': round(y_res.mean(), 4)},
])
samp_df.to_csv(f'{REPORT_DIR}/12_sampling.csv', index=False)
print(samp_df.to_string())
print('\nSaved: 12_sampling.csv')

---
## 5. Evaluation Function

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import mlflow.sklearn

mlflow.set_tracking_uri('http://mlflow:5000')
mlflow.set_experiment('bcp_appetence_unified')

def lift_at_k(y_true, y_proba, k=0.40):
    arr = np.array(y_true)
    idx = np.argsort(y_proba)[::-1]
    n_top = int(len(arr) * k)
    cap = arr[idx[:n_top]].sum() / arr.sum()
    lift = cap / k
    return round(float(lift), 4), round(float(cap), 4)

def evaluate(name, y_true, y_proba, k=LIFT_K):
    lift, cap = lift_at_k(y_true, y_proba, k)
    yp = (y_proba >= 0.5).astype(int)
    return {
        'model':         name,
        'lift_40pct':    lift,
        'capture_40pct': cap,
        'pr_auc':        round(float(average_precision_score(y_true, y_proba)), 4),
        'roc_auc':       round(float(roc_auc_score(y_true, y_proba)), 4),
        'f1':            round(float(f1_score(y_true, yp, zero_division=0)), 4),
    }

results = []
print('Evaluation ready')
print(f'Primary metric: Lift@{int(LIFT_K*100)}%')

---
## 6. Model Training with RFE

In [ ]:
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def get_importances(model, X, y, model_type):
    if model_type == 'lgbm':
        model.fit(X, y, eval_set=[(X, y)],
                  callbacks=[lgb.early_stopping(20), lgb.log_evaluation(-1)])
    elif model_type == 'xgb':
        model.fit(X, y, eval_set=[(X, y)], verbose=False)
    else:
        model.fit(X, y)
    return model.feature_importances_

def rfe_run(factory, model_type, X_smote, y_smote, X_val, y_val_arg,
            name, step=0.1, min_features=10):
    feats = list(X_smote.columns)
    best_lift = -1
    best_feats = feats[:]
    history = []
    rnd = 0
    print(f'\nRFE -- {name} | start={len(feats)} features | step={int(step*100)}%')
    print('-' * 60)
    while len(feats) >= min_features:
        rnd += 1
        model = factory()
        imps = get_importances(model, X_smote[feats], y_smote, model_type)
        yp = model.predict_proba(X_val[feats])[:, 1]
        lift, cap = lift_at_k(y_val_arg, yp)
        pr = round(float(average_precision_score(y_val_arg, yp)), 4)
        history.append({
            'round': rnd, 'n_features': len(feats),
            'lift_40pct': lift, 'capture_40pct': cap, 'pr_auc': pr
        })
        print(f'  R{rnd:>2} | f={len(feats):>3} | lift={lift:.4f} | cap={cap:.4f} | pr={pr:.4f}')
        if lift > best_lift:
            best_lift = lift
            best_feats = feats[:]
        imp_series = pd.Series(imps, index=feats)
        n_drop = max(1, int(len(feats) * step))
        to_drop = imp_series.nsmallest(n_drop).index.tolist()
        feats = [f for f in feats if f not in to_drop]
        if len(feats) < min_features:
            break
    print(f'\nBest: {len(best_feats)} features | Lift@40%={best_lift:.4f}')
    return best_feats, pd.DataFrame(history)

rfe_results = {}
print('RFE pipeline ready')

In [ ]:
# 6.1 LightGBM RFE
def lgbm_factory():
    return lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        objective='binary', random_state=42, n_jobs=-1, verbose=-1
    )

with mlflow.start_run(run_name='LightGBM_RFE'):
    bf, hist = rfe_run(lgbm_factory, 'lgbm', X_res, y_res, X_val_fs, y_val, 'LightGBM')
    fm = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        objective='binary', random_state=42, n_jobs=-1, verbose=-1
    )
    fm.fit(X_res[bf], y_res, eval_set=[(X_val_fs[bf], y_val)],
           callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('LightGBM', y_val, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE'})
    #mlflow.lightgbm.log_model(fm, artifact_path='lgbm')
    hist['model'] = 'LightGBM'
    hist.to_csv(f'{REPORT_DIR}/13a_rfe_lgbm.csv', index=False)
    rfe_results['LightGBM'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal: {m}')

In [ ]:
# 6.2 XGBoost RFE
def xgb_factory():
    return xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='aucpr', early_stopping_rounds=20,
        random_state=42, verbosity=0, n_jobs=-1
    )

with mlflow.start_run(run_name='XGBoost_RFE'):
    bf, hist = rfe_run(xgb_factory, 'xgb', X_res, y_res, X_val_fs, y_val, 'XGBoost')
    fm = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='aucpr', early_stopping_rounds=50,
        random_state=42, verbosity=0, n_jobs=-1
    )
    fm.fit(X_res[bf], y_res, eval_set=[(X_val_fs[bf], y_val)], verbose=False)
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('XGBoost', y_val, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE'})
    #mlflow.xgboost.log_model(fm, artifact_path='xgb')
    hist['model'] = 'XGBoost'
    hist.to_csv(f'{REPORT_DIR}/13b_rfe_xgb.csv', index=False)
    rfe_results['XGBoost'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal: {m}')

In [ ]:
# 6.3 Random Forest RFE
def rf_factory():
    return RandomForestClassifier(
        n_estimators=100, max_depth=10,
        min_samples_leaf=50, n_jobs=-1, random_state=42
    )

with mlflow.start_run(run_name='RandomForest_RFE'):
    bf, hist = rfe_run(rf_factory, 'rf', X_res, y_res, X_val_fs, y_val, 'RandomForest')
    fm = RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_leaf=50, n_jobs=-1, random_state=42
    )
    fm.fit(X_res[bf], y_res)
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('RandomForest', y_val, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE'})
    #mlflow.sklearn.log_model(fm, artifact_path='rf')
    hist['model'] = 'RandomForest'
    hist.to_csv(f'{REPORT_DIR}/13c_rfe_rf.csv', index=False)
    rfe_results['RandomForest'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal: {m}')

In [ ]:
# 6.4 AdaBoost RFE
def ada_factory():
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=100, learning_rate=0.1, random_state=42
    )

with mlflow.start_run(run_name='AdaBoost_RFE'):
    bf, hist = rfe_run(ada_factory, 'ada', X_res, y_res, X_val_fs, y_val, 'AdaBoost')
    fm = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=200, learning_rate=0.1, random_state=42
    )
    fm.fit(X_res[bf], y_res)
    yp = fm.predict_proba(X_val_fs[bf])[:, 1]
    m = evaluate('AdaBoost', y_val, yp)
    m['n_features_rfe'] = len(bf)
    mlflow.log_metrics({k: v for k, v in m.items() if k not in ['model', 'n_features_rfe']})
    mlflow.log_params({'n_features_rfe': len(bf), 'sampling': 'BorderlineSMOTE'})
    #mlflow.sklearn.log_model(fm, artifact_path='ada')
    hist['model'] = 'AdaBoost'
    hist.to_csv(f'{REPORT_DIR}/13d_rfe_ada.csv', index=False)
    rfe_results['AdaBoost'] = {'f': bf, 'model': fm, 'h': hist, 'p': yp, 'm': m}
    results.append(m)
    print(f'\nFinal: {m}')

In [ ]:
# 6.5 RFE summary + convergence plots
rfe_sum = pd.DataFrame([{
    'model': n,
    'features_before': len(final_features),
    'features_after': len(r['f']),
    'lift_40pct': r['m']['lift_40pct'],
    'pr_auc': r['m']['pr_auc']
} for n, r in rfe_results.items()])
rfe_sum.to_csv(f'{REPORT_DIR}/14_rfe_summary.csv', index=False)

pd.DataFrame({
    n: pd.Series(sorted(r['f'])) for n, r in rfe_results.items()
}).to_csv(f'{REPORT_DIR}/15_rfe_features_per_model.csv', index=False)

consensus = sorted(set.intersection(*[set(r['f']) for r in rfe_results.values()]))
pd.DataFrame({'feature': consensus}).to_csv(f'{REPORT_DIR}/16_rfe_consensus.csv', index=False)

print('RFE Summary:')
print(rfe_sum.to_string())
print(f'\nConsensus features ({len(consensus)}): {consensus}')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors = ['steelblue', 'darkorange', 'green', 'red']
for i, (name, res) in enumerate(rfe_results.items()):
    h = res['h'].sort_values('n_features')
    br = h.loc[h['lift_40pct'].idxmax()]
    axes[i].plot(h['n_features'], h['lift_40pct'], color=colors[i], lw=2, marker='o', ms=5)
    axes[i].axvline(x=br['n_features'], color='black', linestyle='--', alpha=0.7,
                    label=f'Best: {int(br["n_features"])} feats')
    axes[i].set_title(f'{name}', fontweight='bold')
    axes[i].set_xlabel('Number of features')
    axes[i].set_ylabel('Lift@40%')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].invert_xaxis()
plt.suptitle('RFE Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/17_rfe_convergence.png', dpi=150)
plt.show()

# Assign variables for downstream
lgbm = rfe_results['LightGBM']['model']
xgbm = rfe_results['XGBoost']['model']
rf   = rfe_results['RandomForest']['model']
ada  = rfe_results['AdaBoost']['model']
y_proba_lgbm = rfe_results['LightGBM']['p']
y_proba_xgb  = rfe_results['XGBoost']['p']
y_proba_rf   = rfe_results['RandomForest']['p']
y_proba_ada  = rfe_results['AdaBoost']['p']
lgbm_feats = rfe_results['LightGBM']['f']
xgb_feats  = rfe_results['XGBoost']['f']
rf_feats   = rfe_results['RandomForest']['f']
ada_feats  = rfe_results['AdaBoost']['f']
print('\nSaved: 14-17 RFE reports')

---
## 7. Benchmark

In [ ]:
rdf = pd.DataFrame(results).sort_values('lift_40pct', ascending=False)
rdf.index = range(1, len(rdf) + 1)
rdf.to_csv(f'{REPORT_DIR}/18_benchmark.csv', index=False)
print('Benchmark (sorted by Lift@40%):')
print(rdf.to_string())
print('\nSaved: 18_benchmark.csv')

---
## 8. Lift & Centile Analysis

In [ ]:
model_probas = {
    'LightGBM':     y_proba_lgbm,
    'XGBoost':      y_proba_xgb,
    'RandomForest': y_proba_rf,
    'AdaBoost':     y_proba_ada,
}

def lift_table(y_true, y_proba, label=''):
    dfl = pd.DataFrame({'y_true': np.array(y_true), 'y_proba': y_proba})
    dfl = dfl.sort_values('y_proba', ascending=False).reset_index(drop=True)
    dfl['decile'] = pd.qcut(dfl.index, 10, labels=False) + 1
    rate = np.array(y_true).mean()
    t = dfl.groupby('decile').agg(
        n=('y_true', 'count'), n_pos=('y_true', 'sum'), avg_score=('y_proba', 'mean')
    ).reset_index()
    t['pos_rate'] = (t['n_pos'] / t['n']).round(4)
    t['lift'] = (t['pos_rate'] / rate).round(4)
    t['cum_cap_pct'] = (t['n_pos'].cumsum() / t['n_pos'].sum() * 100).round(2)
    t['model'] = label
    print(f'\n{label} (rate={rate:.4f}):')
    print(t.to_string(index=False))
    return t

def centile_analysis(y_true, y_proba, label=''):
    arr = np.array(y_true)
    idx = np.argsort(y_proba)[::-1]
    total_pos = arr.sum()
    rate = arr.mean()
    n = len(arr)
    rows = []
    print(f'\nCentile {label}:')
    for c in [1, 2, 5, 10, 20, 30, 40, 50]:
        nt = int(n * c / 100)
        np_c = arr[idx[:nt]].sum()
        cap = np_c / total_pos * 100
        prec = np_c / nt if nt > 0 else 0
        lift = prec / rate if rate > 0 else 0
        mk = ' <-- primary' if c == 40 else ''
        print(f'Top{c:>3}% n={nt:>9,} pos={int(np_c):>7,} cap={cap:>6.1f}% lift={lift:.2f}x{mk}')
        rows.append({
            'model': label, 'centile': c, 'n': nt, 'n_pos': int(np_c),
            'capture_pct': round(cap, 2), 'precision': round(float(prec), 4),
            'lift': round(float(lift), 4)
        })
    return pd.DataFrame(rows)

all_lt = []
all_ct = []
for name, proba in model_probas.items():
    all_lt.append(lift_table(y_val, proba, label=name))
    all_ct.append(centile_analysis(y_val, proba, label=name))

pd.concat(all_lt).to_csv(f'{REPORT_DIR}/19_lift_tables.csv', index=False)
pd.concat(all_ct).to_csv(f'{REPORT_DIR}/20_centile_analysis.csv', index=False)
print('\nSaved: 19_lift_tables.csv  20_centile_analysis.csv')

In [ ]:
# Lift curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['steelblue', 'darkorange', 'green', 'red']
for (name, proba), color in zip(model_probas.items(), colors):
    arr = np.array(y_val)
    idx = np.argsort(proba)[::-1]
    total_pos = arr.sum()
    n = len(arr)
    step = max(1, n // 200)
    pp, pc, lv = [], [], []
    for i in range(1, n + 1, step):
        p = i / n * 100
        c = arr[idx[:i]].sum() / total_pos * 100
        pp.append(p); pc.append(c); lv.append(c / p if p > 0 else 1)
    axes[0].plot(pp, pc, color=color, lw=2, label=name)
    axes[1].plot(pp, lv, color=color, lw=2, label=name)
axes[0].plot([0, 100], [0, 100], '--', color='gray', label='Random')
axes[1].axhline(y=1, color='gray', linestyle='--', label='No lift')
for ax in axes:
    ax.axvline(x=40, color='black', linestyle=':', alpha=0.7, label='40% cutoff')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[0].set(xlabel='% Population', ylabel='% Subscribers', title='Cumulative Gains')
axes[1].set(xlabel='% Population', ylabel='Lift', title='Lift Curve')
plt.suptitle('Lift Analysis -- Unified Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/21_lift_curves.png', dpi=150)
plt.show()
print('Saved: 21_lift_curves.png')

---
## 9. Ensemble

In [ ]:
# Simple average
ens_avg = (y_proba_lgbm + y_proba_xgb + y_proba_rf + y_proba_ada) / 4
with mlflow.start_run(run_name='Ensemble_Average'):
    m = evaluate('Ensemble_Average', y_val, ens_avg)
    mlflow.log_metrics({k: v for k, v in m.items() if k != 'model'})
    results.append(m)
    print(m)

# Weighted average
lift_scores = {n: r['m']['lift_40pct'] for n, r in rfe_results.items()}
total_lift = sum(lift_scores.values())
weights = {k: v / total_lift for k, v in lift_scores.items()}
print('\nWeights:', {k: round(v, 3) for k, v in weights.items()})

ens_w = (
    weights['LightGBM']     * y_proba_lgbm +
    weights['XGBoost']      * y_proba_xgb  +
    weights['RandomForest'] * y_proba_rf   +
    weights['AdaBoost']     * y_proba_ada
)
with mlflow.start_run(run_name='Ensemble_Weighted'):
    m = evaluate('Ensemble_Weighted', y_val, ens_w)
    mlflow.log_metrics({k: v for k, v in m.items() if k != 'model'})
    results.append(m)
    print(m)

# Stacking (sample meta-features to avoid memory issues)
X_train_meta_sample = X_train_fs.sample(n=SAMPLE_N, random_state=42)
y_train_meta_sample = y_train.loc[X_train_meta_sample.index]
meta_tr = np.column_stack([
    lgbm.predict_proba(X_train_meta_sample[lgbm_feats])[:, 1],
    xgbm.predict_proba(X_train_meta_sample[xgb_feats])[:, 1],
    rf.predict_proba(X_train_meta_sample[rf_feats])[:, 1],
    ada.predict_proba(X_train_meta_sample[ada_feats])[:, 1],
])
meta_vl = np.column_stack([y_proba_lgbm, y_proba_xgb, y_proba_rf, y_proba_ada])

meta_model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=15,
    objective='binary', random_state=42, n_jobs=-1, verbose=-1
)
meta_model.fit(meta_tr, y_train_meta_sample)
ens_s = meta_model.predict_proba(meta_vl)[:, 1]

del X_train_meta_sample, y_train_meta_sample, meta_tr
gc.collect()

with mlflow.start_run(run_name='Ensemble_Stacking'):
    m = evaluate('Ensemble_Stacking', y_val, ens_s)
    mlflow.log_metrics({k: v for k, v in m.items() if k != 'model'})
    results.append(m)
    print(m)

In [ ]:
# Final benchmark
fdf = pd.DataFrame(results).sort_values('lift_40pct', ascending=False)
fdf.index = range(1, len(fdf) + 1)
fdf.to_csv(f'{REPORT_DIR}/22_final_benchmark.csv', index=False)

pd.DataFrame([{
    'model': k,
    'lift_40pct': lift_scores.get(k),
    'weight': round(weights.get(k, 0), 4)
} for k in rfe_results]).to_csv(f'{REPORT_DIR}/23_ensemble_weights.csv', index=False)

print('FINAL BENCHMARK:')
print(fdf.to_string())
print('\nSaved: 22_final_benchmark.csv  23_ensemble_weights.csv')

---
## 10. Scoring on full 3.2M client base

In [ ]:
# Score the full base by chunks to avoid memory crashes
CHUNK_SIZE = 500_000

pdf_full = pd.read_parquet('/home/jovyan/work/master_preprocessed')
print(f'Full dataset: {len(pdf_full):,} clients')

n = len(pdf_full)
score_lgbm = np.zeros(n)
score_xgb  = np.zeros(n)
score_rf   = np.zeros(n)
score_ada  = np.zeros(n)

for start in range(0, n, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n)
    print(f'  Scoring chunk {start:,} -> {end:,}')
    chunk = pdf_full.iloc[start:end]
    score_lgbm[start:end] = lgbm.predict_proba(chunk[lgbm_feats].fillna(0))[:, 1]
    score_xgb[start:end]  = xgbm.predict_proba(chunk[xgb_feats].fillna(0))[:, 1]
    score_rf[start:end]   = rf.predict_proba(chunk[rf_feats].fillna(0))[:, 1]
    score_ada[start:end]  = ada.predict_proba(chunk[ada_feats].fillna(0))[:, 1]

pdf_full['score_lgbm'] = score_lgbm
pdf_full['score_xgb']  = score_xgb
pdf_full['score_rf']   = score_rf
pdf_full['score_ada']  = score_ada
pdf_full['score_ensemble'] = (
    weights['LightGBM']     * score_lgbm +
    weights['XGBoost']      * score_xgb  +
    weights['RandomForest'] * score_rf   +
    weights['AdaBoost']     * score_ada
)
pdf_full['rank']   = pdf_full['score_ensemble'].rank(ascending=False).astype(int)
pdf_full['decile'] = pd.qcut(pdf_full['score_ensemble'], q=10, labels=False) + 1

score_cols = ['score_lgbm', 'score_xgb', 'score_rf', 'score_ada', 'score_ensemble']
out = pdf_full[LABELS + score_cols + ['rank', 'decile']].sort_values('score_ensemble', ascending=False)
out.to_parquet('/home/jovyan/work/scores_unified.parquet', index=False)

# Add unified label for decile analysis
out['label_any'] = ((out['label_maRetraite'] == 1) |
                    (out['label_avenirMesEnfants'] == 1) |
                    (out['label_epargneEvolution'] == 1)).astype(int)

dec = out.groupby('decile').agg(
    n=('score_ensemble', 'count'),
    avg_score=('score_ensemble', 'mean'),
    subs=('label_any', 'sum')
).reset_index()
dec['rate'] = (dec['subs'] / dec['n']).round(4)
dec.to_csv(f'{REPORT_DIR}/24_decile_distribution.csv', index=False)

print(f'\nScored {len(pdf_full):,} clients')
print(dec.to_string())
print('\nSaved: scores_unified.parquet  24_decile_distribution.csv')

del pdf_full, out, score_lgbm, score_xgb, score_rf, score_ada
gc.collect()

---
## 11. SHAP (on sample)

In [ ]:
import shap

X_shap = X_val_fs[lgbm_feats].sample(5000, random_state=42)
print(f'SHAP sample: {X_shap.shape}')

explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_shap)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_df = pd.DataFrame({
    'feature': X_shap.columns,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
shap_df.to_csv(f'{REPORT_DIR}/25_shap.csv', index=False)

print('\nTop 20 by |SHAP|:')
print(shap_df.head(20).to_string())

shap.summary_plot(sv, X_shap, max_display=20, show=False)
plt.title('SHAP Summary -- Unified Label')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/26_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

shap.summary_plot(sv, X_shap, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Bar -- Unified Label')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/27_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved: 25-27 shap files')

In [ ]:
# Final inventory
import glob
files = sorted(glob.glob(f'{REPORT_DIR}/*'))
print(f'=== {len(files)} report files ===')
for f in files:
    size = os.path.getsize(f)
    ftype = 'CSV' if f.endswith('.csv') else 'PNG' if f.endswith('.png') else 'OTHER'
    print(f'  [{ftype}] {os.path.basename(f):<50} {size:>8,} bytes')